(model-training-tutorial)=
# Train, compare, and register models

This notebook provides a quick overview of training ML models using [MLRun](https://www.mlrun.org/) AI orchestration framework.

Make sure you reviewed the basics in MLRun [**Quick Start Tutorial**](01-mlrun-basics.ipynb).

Tutorial steps:
- [**Define an MLRun project and a training functions**](#define-mlrun-project-and-a-training-functions)
- [**Run the function, log the artifacts and model**](#run-the-training-function-and-log-the-artifacts-and-model)
- [**Hyper-parameter tuning and model/experiment comparison**](#hyper-parameter-tuning-and-modelexperiment-comparison)
- [**Build and test the model serving functions**](#build-and-test-the-model-serving-functions)

<iframe width="560" height="315" src="https://www.youtube.com/embed/bZgBsmLMdQo" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe>

## MLRun installation and configuration

Before running this notebook make sure `mlrun` and `sklearn` packages are installed (`pip install mlrun scikit-learn~=1.5.1`) and that you have configured the access to the MLRun service. 

In [ ]:
# Install MLRun if not installed, run this only once (restart the notebook after the install !!!)
%pip install mlrun

<a id="define-project"></a>
## Define MLRun project and a training functions

You should create, load, or use (get) an [MLRun project](https://docs.mlrun.org/en/stable/projects/project.html) that holds all your functions and assets.

**Get or create a new project**

The `get_or_create_project()` method tries to load the project from MLRun DB. If the project does not exist, it creates a new one.

In [ ]:
import mlrun

project = mlrun.get_or_create_project("tutorial", context="./", user_project=True)

**Add (auto) MLOps to your training function**

Training functions generate models and various model statistics. You'll want to store the models along with all the relevant data,
metadata, and measurements. MLRun can apply all the MLOps functionality automatically ("Auto-MLOps") by simply using the framework-specific [`apply_mlrun()`](https://docs.mlrun.org/en/stable/api/mlrun.frameworks/mlrun.frameworks.auto_mlrun.html#module-mlrun.frameworks.auto_mlrun.auto_mlrun) method.

This is the line to add to your code, as shown in the training function below. 

```python
apply_mlrun(model=model, model_name="my_model", x_test=x_test, y_test=y_test)
```

`apply_mlrun()` manages the training process and automatically logs all the framework-specific model object, details, data, metadata, and metrics.
It accepts the model object and various optional parameters. When specifying the `x_test` and `y_test` data it generates various plots and calculations to evaluate the model.
Metadata and parameters are automatically recorded (from MLRun `context` object) and therefore don't need to be specified.

**Function code**

Run the following cell to generate the `trainer.py` file (or copy it manually):

In [3]:
%%writefile src/trainer.py

import pandas as pd

from sklearn import ensemble
from sklearn.model_selection import train_test_split

import mlrun
from mlrun.frameworks.sklearn import apply_mlrun


def train(
    dataset: pd.DataFrame,
    label_column: str = "label",
    n_estimators: int = 100,
    learning_rate: float = 0.1,
    max_depth: int = 3,
    model_name: str = "cancer_classifier",
):
    # Initialize the x & y data
    x = dataset.drop(label_column, axis=1)
    y = dataset[label_column]

    # Train/Test split the dataset
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.2, random_state=42
    )

    # Pick an ideal ML model
    model = ensemble.GradientBoostingClassifier(
        n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth
    )

    # -------------------- The only line you need to add for MLOps -------------------------
    # Wraps the model with MLOps (test set is provided for analysis & accuracy measurements)
    apply_mlrun(model=model, model_name=model_name, x_test=x_test, y_test=y_test)
    # --------------------------------------------------------------------------------------

    # Train the model
    model.fit(x_train, y_train)


Overwriting src/trainer.py


**Create a serverless function object from the code above, and register it in the project**

In [4]:
trainer = project.set_function(
    "src/trainer.py", name="trainer", kind="job", image="mlrun/mlrun", handler="train"
)

<a id="run-function"></a>
## Run the training function and log the artifacts and model

**Create a dataset for training**

In [5]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

breast_cancer = load_breast_cancer()
breast_cancer_dataset = pd.DataFrame(
    data=breast_cancer.data, columns=breast_cancer.feature_names
)
breast_cancer_labels = pd.DataFrame(data=breast_cancer.target, columns=["label"])
breast_cancer_dataset = pd.concat([breast_cancer_dataset, breast_cancer_labels], axis=1)

breast_cancer_dataset.to_csv("cancer-dataset.csv", index=False)

**Run the function (locally) using the generated dataset**

In [6]:
trainer_run = project.run_function(
    "trainer",
    inputs={"dataset": "cancer-dataset.csv"},
    params={"n_estimators": 100, "learning_rate": 1e-1, "max_depth": 3},
    local=True,
)

> 2025-05-15 17:23:56,782 [info] Storing function: {"db":"https://mlrun-api.default-tenant.app.innovation-dev.iguazio-cd2.com","name":"trainer-train","uid":"a15da8a2ac4a470fab58c98dbbd26b77"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifact_uris
tutorial-shapira,...bbd26b77,0,May 15 14:23:57,NaT,completed,run,trainer-train,v3io_user=shapirakind=localowner=shapirahost=M-QXN63PHMF9,dataset,n_estimators=100learning_rate=0.1max_depth=3,accuracy=0.956140350877193f1_score=0.965034965034965precision_score=0.9583333333333334recall_score=0.971830985915493,feature-importance=store://artifacts/tutorial-shapira/trainer-train_feature-importance#0@a15da8a2ac4a470fab58c98dbbd26b77^0340e079307dadc3a18a8e5568eb7ab1da9d6bfbtest_set=store://datasets/tutorial-shapira/trainer-train_test_set#0@a15da8a2ac4a470fab58c98dbbd26b77^3fd834c8bd412b9f534dffbec57f48fdfc52e673confusion-matrix=store://artifacts/tutorial-shapira/trainer-train_confusion-matrix#0@a15da8a2ac4a470fab58c98dbbd26b77^0fe7e29d87394feb2bf1fedca364740f11b878fcroc-curves=store://artifacts/tutorial-shapira/trainer-train_roc-curves#0@a15da8a2ac4a470fab58c98dbbd26b77^e0142cda96cae30a1195a126eb0550abce6059a9calibration-curve=store://artifacts/tutorial-shapira/trainer-train_calibration-curve#0@a15da8a2ac4a470fab58c98dbbd26b77^5bbe679c8c5b054594366f5a757d139f50c769dbmodel=store://models/tutorial-shapira/cancer_classifier#0@a15da8a2ac4a470fab58c98dbbd26b77^47e773696bcb81cc69bc219fb28166467c77de76


> 2025-05-15 17:24:19,639 [info] Run execution finished: {"name":"trainer-train","status":"completed"}


<br>

**View the auto generated results and artifacts**

In [7]:
trainer_run.outputs

{'accuracy': 0.956140350877193,
 'f1_score': 0.965034965034965,
 'precision_score': 0.9583333333333334,
 'recall_score': 0.971830985915493,
 'feature-importance': 'v3io:///projects/tutorial-shapira/artifacts/trainer-train/0/feature-importance.html',
 'test_set': 'store://datasets/tutorial-shapira/trainer-train_test_set:latest@a15da8a2ac4a470fab58c98dbbd26b77^3fd834c8bd412b9f534dffbec57f48fdfc52e673',
 'confusion-matrix': 'v3io:///projects/tutorial-shapira/artifacts/trainer-train/0/confusion-matrix.html',
 'roc-curves': 'v3io:///projects/tutorial-shapira/artifacts/trainer-train/0/roc-curves.html',
 'calibration-curve': 'v3io:///projects/tutorial-shapira/artifacts/trainer-train/0/calibration-curve.html',
 'model': 'store://models/tutorial-shapira/cancer_classifier:latest@a15da8a2ac4a470fab58c98dbbd26b77^47e773696bcb81cc69bc219fb28166467c77de76'}

In [8]:
trainer_run.artifact("feature-importance").show()

**Export model files + metadata into a zip** (requires MLRun 1.1.0 and later)

You can `export()` the model package (files + metadata) into a zip, and load it on a remote system/cluster by running `model = project.import_artifact(key, path)`). 

In [9]:
trainer_run.artifact("model").meta.export("src/model.zip")

<a id="hyper-param"></a>
## Hyper-parameter tuning and model/experiment comparison

Run a `GridSearch` with a couple of parameters, and select the best run with respect to the `max accuracy`. <br>
(For more details, see MLRun [Hyper-Param and Iterative jobs](https://docs.mlrun.org/en/stable/hyper-params.html).)

For basic usage you can run the hyperparameters tuning job by using the arguments: 
* `hyperparams` for the hyperparameters options and values of choice.
* `selector` for specifying how to select the best model.

**Running a remote function**

To run the hyper-param task over the cluster you need the input data to be available for the job, using object storage or the MLRun versioned artifact store.

The following line logs (and uploads) the dataframe as a project artifact:

In [10]:
dataset_artifact = project.log_dataset(
    "cancer-dataset", df=breast_cancer_dataset, index=False
)

Run the function over the remote Kubernetes cluster (`local` is not set):

In [12]:
hp_tuning_run = project.run_function(
    "trainer",
    inputs={"dataset": dataset_artifact.uri},
    hyperparams={
        "n_estimators": [10, 100, 1000],
        "learning_rate": [1e-1, 1e-3],
        "max_depth": [2, 8],
    },
    selector="max.accuracy",
)

> 2025-05-15 17:41:11,373 [info] Storing function: {"db":"https://mlrun-api.default-tenant.app.innovation-dev.iguazio-cd2.com","name":"trainer-train","uid":"cdd5e740ff8c4e58880b8683aaa69f61"}
> 2025-05-15 17:41:12,758 [info] Job is running in the background, pod: trainer-train-f87t9
> 2025-05-15 14:42:06,004 [info] Best iteration=3, used criteria max.accuracy
> 2025-05-15 14:42:06,880 [info] To track results use the CLI: {"info_cmd":"mlrun get run cdd5e740ff8c4e58880b8683aaa69f61 -p tutorial-shapira","logs_cmd":"mlrun logs cdd5e740ff8c4e58880b8683aaa69f61 -p tutorial-shapira"}
> 2025-05-15 14:42:06,880 [info] Or click for UI: {"ui_url":"https://dashboard.default-tenant.app.innovation-dev.iguazio-cd2.com/mlprojects/tutorial-shapira/jobs/monitor-jobs/trainer-train/cdd5e740ff8c4e58880b8683aaa69f61/overview"}
> 2025-05-15 14:42:06,881 [info] Run execution finished: {"name":"trainer-train","status":"completed"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifacts
tutorial-shapira,...aaa69f61,0,May 15 14:41:22,2025-05-15 14:42:06.845818+00:00,completed,run,trainer-train,v3io_user=shapirakind=jobowner=shapiramlrun/client_version=1.8.0-rc45mlrun/client_python_version=3.9.22,dataset,,best_iteration=3accuracy=0.9649122807017544f1_score=0.9722222222222222precision_score=0.958904109589041recall_score=0.9859154929577465,parallel_coordinatesiteration_resultsmodelcalibration-curveroc-curvesconfusion-matrixtest_setfeature-importance


> 2025-05-15 17:42:11,883 [info] Run execution finished: {"name":"trainer-train","status":"completed"}


<br>

**View Hyper-param results and the selected run in the MLRun UI**

![hprun](./_static/images/hprun.png)

**Interactive Parallel Coordinates Plot**

![pcp](./_static/images/pcp.png)

<br>

**List the generated models and compare the different runs**

In [ ]:
hp_tuning_run.outputs

In [ ]:
# List the models in the project (can apply filters)
models = project.list_models()
for model in models:
    print(f"uri: {model.uri}, metrics: {model.metrics}")

In [ ]:
# To view the full model object use:
# print(models[0].to_yaml())

In [ ]:
# Compare the runs (generate interactive parallel coordinates plot and a table)
project.list_runs(name="trainer-train", iter=True).compare()

<a id="model-serving"></a>
## Build and test the model serving functions

MLRun serving can produce managed, real-time, serverless, pipelines composed of various data processing and ML tasks. The pipelines use the Nuclio real-time serverless engine, which can be deployed anywhere. For more details and examples, see the [MLRun Serving Graphs](https://docs.mlrun.org/en/stable/serving/serving-graph.html).

**Create a model serving function from your [code](src/serving.py), and [(view it here)](03-model-serving.ipynb)**

In [ ]:
serving_fn = project.set_function(
    func="",
    name="serving",
    image="mlrun/mlrun",
    kind="serving",
)
serving_fn.add_model(
    "cancer-classifier",
    model_path=hp_tuning_run.outputs["model"],
    class_name="mlrun.frameworks.sklearn.SklearnModelServer",
)

In [ ]:
# Create a mock (simulator of the real-time function)
server = serving_fn.to_mock_server()

my_data = {
    "inputs": [
        [
            1.371e01,
            2.083e01,
            9.020e01,
            5.779e02,
            1.189e-01,
            1.645e-01,
            9.366e-02,
            5.985e-02,
            2.196e-01,
            7.451e-02,
            5.835e-01,
            1.377e00,
            3.856e00,
            5.096e01,
            8.805e-03,
            3.029e-02,
            2.488e-02,
            1.448e-02,
            1.486e-02,
            5.412e-03,
            1.706e01,
            2.814e01,
            1.106e02,
            8.970e02,
            1.654e-01,
            3.682e-01,
            2.678e-01,
            1.556e-01,
            3.196e-01,
            1.151e-01,
        ]
    ]
}
server.test("/v2/models/cancer-classifier/infer", body=my_data)

## Done!

Congratulations! You've completed Part 2 of the MLRun getting-started tutorial.
Proceed to [**Part 3: Serving pre-trained ML/DL models**](03-model-serving.ipynb) to learn how to deploy and serve your model using a serverless function.